<a href="https://colab.research.google.com/github/joemacKE/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Rainfall Analysis and Planting Advisory
Project: PLP Data Analysis Assignment 2
Repository: plp-da-a2-rainfall-advisory
This notebook cleans the manually entered Machakos rainfal readings, combines the five counties into a NumPy array. analyses rainfall patterns, recommends two-month planting windows, and exports a county-level summary

1. Important libraries and define the dataset
The csv module is part of Python's standard library. NumPy is used for array operations, aggregation, Boolean masks, and identifying maximu values.

In [46]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
] #machakos raw data as received from the field


## 2. Clean the Machakos rainfall readings

The function strips surrounding whitespace and converts valid entries to integers. It catches the specific `ValueError` raised when text such as `"N/A"` or an empty string cannot be converted to an integer. Invalid entries are replaced with the rounded mean of the valid Machakos readings.

In [47]:
def clean_machakos_value(value):
  try:
    return int(value.strip())
  except ValueError:
    return None

machakos_values = [clean_machakos_value(value) for value in machakos_raw]
valid_machakos_values = [value for value in machakos_values if value is not None]
machakos_imputation = round(np.mean(valid_machakos_values))

# replace missing/unparseable values with the rounded valid-value average
machakos_clean = [
    machakos_imputation if value is None else value
    for value in machakos_values
]

imputed_months = [
    month
    for month, value in zip(months, machakos_values)
    if value is None
]

print("Valid Machakos Readings: ", valid_machakos_values)
print("Rounded imputation values: ", machakos_imputation, "mm")
print("Imputed Months: ", imputed_months)
print("Clean Machakos Row: ", machakos_clean)

Valid Machakos Readings:  [48, 35, 88, 140, 20, 9, 12, 45, 182, 96]
Rounded imputation values:  68 mm
Imputed Months:  ['April', 'September']
Clean Machakos Row:  [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Combine the five counties into a NumPy array

The four already-clean rows and the cleaned Machakos row are combined into one two-dimensional array. The array should contain five county rows and twelve monthly columns.
all_rows = clean_rows + [machakos_clean]
R = np.array(all_rows, dtype=int)

print("Rainfall array:")
print(R)
print("Shape:", R.shape)
print("Data type:", R.dtype)

In [48]:
all_rows = clean_rows + [machakos_clean]
R = np.array(all_rows, dtype=int)

print("Rainfall array:")
print(R)
print("Shape:", R.shape)
print("Data type:", R.dtype)

Rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Data type: int64


In [49]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual Rainfall Totals (mm): ")
for county, total in zip(counties, annual_totals):
  print(f"{county}: {total} mm")

print("\nAverage rainfall by month across all counties (mm):")
for month, average in zip(months, monthly_averages):
  print(f"{month}: {average:.1f} mm")


Annual Rainfall Totals (mm): 
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall by month across all counties (mm):
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## 5. Wettest month for each county

`np.argmax(..., axis=1)` returns the column index of the highest monthly rainfall in each county's row. `zip()` pairs each county with its wettest-month result.

In [50]:
wettest_month_indices = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_month_indices]

print("Wettest month by county:")
for county, wettest_month in zip(counties, wettest_months):
    print(f"{county}: {wettest_month}")

Wettest month by county:
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## 6. Count dry months

A dry month is defined in this assignment as a month with rainfall strictly below 50 mm. The Boolean mask identifies those values, and summing across each row counts the dry months per county.

In [51]:
dry_months_mask = R < 50
dry_months_count = dry_months_mask.sum(axis=1)

print("Dry Months Counts (Below 50 mm):")
for county, count in zip(counties, dry_months_count):
  print(f"{county}: {count}")

Dry Months Counts (Below 50 mm):
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## 7. Identify the best consecutive two-month planting window

The shifted-array technique adds each month to the month immediately after it. `np.argmax(..., axis=1)` identifies the starting month of the consecutive pair with the highest combined rainfall for each county.

In [52]:
two_month_totals = R[:, :-1] + R[:, 1:]
planting_window_indices = np.argmax(two_month_totals, axis=1)

planting_windows = [
    f"{months[index]} - {months[index + 1]}"
    for index in planting_window_indices
]

planting_window_rainfall = two_month_totals[
    np.arange(R.shape[0]), planting_window_indices
]

print("Recommended two-month planting windows")
for county, window, total in zip(counties, planting_windows, planting_window_rainfall):
  print(f"county: {window} {total} mm combined")

Recommended two-month planting windows
county: April - May 395 mm combined
county: April - May 265 mm combined
county: April - May 420 mm combined
county: April - May 390 mm combined
county: November - December 278 mm combined


## 8. Export the rainfall summary to CSV

The CSV contains exactly one row per county and the five required columns. The `csv.DictWriter` writes the header and county records to `rainfall_summary.csv`.

In [53]:
summary_rows = [
    {"county": county, "annual_total_mm": int(total), "wettest_month": wettest_month,
     "dry_months": int(dry_count), "planting_window": planting_window}

    for county, total, wettest_month, dry_count, planting_window
    in zip(
        counties, annual_totals, wettest_months, dry_months_count, planting_windows
        )
]

fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csv_file:
  writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
  writer.writeheader()
  writer.writerows(summary_rows)

print("Created rainfall_summary.csv")
print("Number of county records: ", len(summary_rows))

Created rainfall_summary.csv
Number of county records:  5


## Planting Advisory

The rainfall analysis suggests that the NGO should use the strongest consecutive two-month rainfall period in each county as an initial planting-window signal: Nairobi (April-May), Nakuru (April-May), Kisumu (April-May), Mombasa (April-May), and Machakos (November-December). Farmers should prepare land and obtain suitable seed before the recommended window, while confirming local soil moisture and the onset of dependable rain before planting. Counties with several dry months should receive additional drought-risk messaging and guidance on water conservation or locally suitable, shorter-maturing crops. The advisory is based on monthly totals and does not show whether rainfall arrived steadily or in a few intense events. Machakos also contains two manually entered invalid readings, which were imputed using the rounded mean of its valid readings, so its totals and planting recommendation should be treated as estimates. The NGO should validate station records and combine this analysis with current seasonal forecasts and local agricultural extension advice before sending SMS recommendations.